<a href="https://colab.research.google.com/github/theYahyaturk/category/blob/main/ecommerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================
# CELL 1 — INSTALL / IMPORT
# ================================

!pip -q install pandas numpy scikit-learn joblib openpyxl

import pandas as pd
import numpy as np
import joblib
import os
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
# ================================
# CELL 2 — UPLOAD DATASET
# ================================

from google.colab import files
import os

# Expected filename based on previous successful uploads and user input
EXPECTED_FILE_NAME = 'alifeo_ecommerce_ml_dataset_150k_with_negative_data.csv'

try:
    # Attempt to read the file assuming it's already in the Colab environment
    df = pd.read_csv(EXPECTED_FILE_NAME)
    FILE_NAME = EXPECTED_FILE_NAME  # Set FILE_NAME if successful
    print(f"Using existing file: {FILE_NAME}")
except FileNotFoundError:
    # If the file is not found, prompt the user to upload it
    print(f"File '{EXPECTED_FILE_NAME}' not found. Please upload the dataset.")
    uploaded = files.upload()
    FILE_NAME = list(uploaded.keys())[0]  # Get the name of the newly uploaded file
    df = pd.read_csv(FILE_NAME)  # Read the newly uploaded file

print("Dataset Shape:", df.shape)
display(df.head())

Using existing file: alifeo_ecommerce_ml_dataset_150k_with_negative_data.csv
Dataset Shape: (150000, 11)


,product_id,title,description,category,subcategory,brand,variant,pack_size,price_inr,search_keywords,is_product
0,ALF1009893,gloves,gloves is a cricket product in sports & fitnes...,Sports & Fitness,Cricket,Generic,White / Large,20 pair,8979.64,"gloves, cricket, sports & fitness, sports, fit...",True
1,ALF058968,Compact A-Line Kurti Beige,"Catalog listing for a-line kurti, categorized ...",Clothing,Women Kurtis,Generic,Beige / Standard,Standard,4416.00,"a-line kurti, women kurtis, clothing, compact,...",True
2,NEG027325,Non product text for classification example 25...,Non product text for classification example 25...,Not a Product,Not a Product,NaN,NaN,NaN,NaN,NaN,False
3,NEG040510,Non product text for classification example 38...,Non product text for classification example 38...,Not a Product,Not a Product,NaN,NaN,NaN,NaN,NaN,False
4,NEG044298,This entry contains no product details example...,This entry contains no product details example...,Not a Product,Not a Product,NaN,NaN,NaN,NaN,NaN,False


In [3]:
print(df.shape)
print(df.info)


(150000, 11)
<bound method DataFrame.info of         product_id                                              title  \
0       ALF1009893                                             gloves   
1        ALF058968                         Compact A-Line Kurti Beige   
2        NEG027325  Non product text for classification example 25...   
3        NEG040510  Non product text for classification example 38...   
4        NEG044298  This entry contains no product details example...   
...            ...                                                ...   
149995   NEG019880  Non product text for classification example 17...   
149996   NEG003695  Non product text for classification example 1447.   
149997   NEG031933  This entry contains no product details example...   
149998   NEG046868  This entry contains no product details example...   
149999   NEG021959       This is not a product listing example 19711.   

                                              description          category  \

In [ ]:
df.dtypes


,0
product_id,object
title,object
description,object
category,object
subcategory,object
brand,object
variant,object
pack_size,object
price_inr,float64
search_keywords,object


In [4]:
# Check for missing values
print("Missing values per column:")
display(df.isnull().sum())

Missing values per column:


,0
product_id,0
title,0
description,0
category,0
subcategory,0
brand,50000
variant,50000
pack_size,50000
price_inr,51270
search_keywords,50000


In [5]:
print(
    df[df["category"] == "Not a Product"][
        ["title", "description", "search_keywords", "category", "subcategory"]
    ].head(20)
)

                                                title  \
2   Non product text for classification example 25...   
3   Non product text for classification example 38...   
4   This entry contains no product details example...   
17  Non product text for classification example 7262.   
25         General message for testing example 32233.   
26                             I am the founder only.   
32        This is not a product listing example 1421.   
33                             happy birthday thanks.   
36        This is not a product listing example 8501.   
44   No product information is provided example 5454.   
48  This entry contains no product details example...   
49          General message for testing example 6413.   
50  This entry contains no product details example...   
55           General message for testing example 393.   
59        This is not a product listing example 9516.   
61  This entry contains no product details example...   
63  No product information is p

In [7]:
# ================================
# CELL 4 — CATEGORY TRAIN/TEST SPLIT
# ================================

# Create combined text
df["text"] = (
    df["title"].fillna("").astype(str) + " " +
    df["description"].fillna("").astype(str) + " " +
    df["search_keywords"].fillna("").astype(str)
).str.strip()

# Category target
y = df["category"]
X = df["text"]

# Check Product vs Not a Product
print("Total records:", len(df))
print("\nCategory distribution:")
print(y.value_counts())

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining:", len(X_train))
print("Testing :", len(X_test))

# Check that Not a Product exists in both sets
print("\nNOT A PRODUCT records:")
print("Train:", (y_train == "Not a Product").sum())
print("Test :", (y_test == "Not a Product").sum())

Total records: 150000

Category distribution:
category
Not a Product               50000
Medical & Pharmacy          12687
Clothing                     8804
Home Decor                   7842
Home & Kitchen               7026
Electronics                  6222
Footwear                     5169
Grocery                      5101
Beauty & Personal Care       4902
Sports & Fitness             4328
Toys & Games                 4118
Jewelry & Accessories        4064
Books & Stationery           3496
Tools & Home Improvement     3488
Baby                         3448
Pet Supplies                 3441
Automotive                   3129
Office Products              2869
Musical Instruments          2862
Garden & Outdoor             2861
Travel & Luggage             2292
Health & Wellness            1851
Name: count, dtype: int64

Training: 120000
Testing : 30000

NOT A PRODUCT records:
Train: 40000
Test : 10000


In [8]:
# ================================
# CELL 5 — CATEGORY TF-IDF
# ================================

from sklearn.feature_extraction.text import TfidfVectorizer

category_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    max_features=300000
)

# Fit ONLY on training data
X_train_tfidf = category_vectorizer.fit_transform(X_train)

# Transform test data using the same vectorizer
X_test_tfidf = category_vectorizer.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape :", X_test_tfidf.shape)
print("Vocabulary size      :", len(category_vectorizer.vocabulary_))

TF-IDF training shape: (120000, 49447)
TF-IDF testing shape : (30000, 49447)
Vocabulary size      : 49447


In [9]:
# ================================
# CELL 6 — TRAIN CATEGORY MODEL
# ================================

from sklearn.svm import LinearSVC

category_model = LinearSVC(
    C=2.0,
    class_weight="balanced",
    random_state=42
)

category_model.fit(
    X_train_tfidf,
    y_train
)

print("✅ Category model training completed.")
print("Classes:", len(category_model.classes_))
print("\nClasses:")
print(category_model.classes_)

✅ Category model training completed.
Classes: 22

Classes:
['Automotive' 'Baby' 'Beauty & Personal Care' 'Books & Stationery'
 'Clothing' 'Electronics' 'Footwear' 'Garden & Outdoor' 'Grocery'
 'Health & Wellness' 'Home & Kitchen' 'Home Decor' 'Jewelry & Accessories'
 'Medical & Pharmacy' 'Musical Instruments' 'Not a Product'
 'Office Products' 'Pet Supplies' 'Sports & Fitness'
 'Tools & Home Improvement' 'Toys & Games' 'Travel & Luggage']


In [10]:
# ================================
# CELL 7 — CATEGORY EVALUATION
# ================================

from sklearn.metrics import accuracy_score, classification_report

y_pred_category = category_model.predict(X_test_tfidf)

accuracy = accuracy_score(
    y_test,
    y_pred_category
)

print("CATEGORY ACCURACY:", round(accuracy, 4))

print("\nCLASSIFICATION REPORT:")
print(
    classification_report(
        y_test,
        y_pred_category,
        zero_division=0
    )
)

CATEGORY ACCURACY: 1.0

CLASSIFICATION REPORT:
                          precision    recall  f1-score   support

              Automotive       1.00      1.00      1.00       626
                    Baby       1.00      1.00      1.00       690
  Beauty & Personal Care       1.00      1.00      1.00       981
      Books & Stationery       1.00      1.00      1.00       699
                Clothing       1.00      1.00      1.00      1761
             Electronics       1.00      1.00      1.00      1244
                Footwear       1.00      1.00      1.00      1034
        Garden & Outdoor       1.00      1.00      1.00       572
                 Grocery       1.00      1.00      1.00      1020
       Health & Wellness       1.00      1.00      1.00       370
          Home & Kitchen       1.00      1.00      1.00      1405
              Home Decor       1.00      1.00      1.00      1568
   Jewelry & Accessories       1.00      1.00      1.00       813
      Medical & Pharmacy    

In [11]:
# ================================
# CELL 8 — SUBCATEGORY DATA PREPARATION
# ================================

# Not a Product records ko subcategory training se remove karo
product_df = df[
    df["category"] != "Not a Product"
].copy()

# Empty subcategory records remove karo
product_df = product_df[
    product_df["subcategory"].str.strip() != ""
].copy()

# Text ensure karo
product_df["text"] = (
    product_df["title"].fillna("").astype(str) + " " +
    product_df["description"].fillna("").astype(str) + " " +
    product_df["search_keywords"].fillna("").astype(str)
).str.strip()

product_df = product_df[
    product_df["text"].str.len() > 0
].copy()

print("Product records:", len(product_df))
print("Categories:", product_df["category"].nunique())
print("Subcategories:", product_df["subcategory"].nunique())

print("\nTop subcategories:")
print(product_df["subcategory"].value_counts().head(20))

Product records: 100000
Categories: 21
Subcategories: 184

Top subcategories:
subcategory
Kitchen Appliances      787
Furniture               736
Cookware                736
Bedding                 684
Bathroom Accessories    683
Cleaning Supplies       681
Home Lighting           681
Dining & Serveware      680
Mobile Accessories      679
Skincare                679
Computer Accessories    679
Kitchen Tools           679
Kitchen Storage         679
Beverages               678
Fitness Equipment       677
Dry Fruits & Nuts       677
Makeup                  676
Snacks                  671
Mobile Phones           635
Laptops                 630
Name: count, dtype: int64


In [12]:
# ================================
# CELL 9 — SUBCATEGORY DISTRIBUTION
# ================================

subcategory_counts = (
    product_df
    .groupby(["category", "subcategory"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(subcategory_counts.head(50))

,category,subcategory,count
84,Home & Kitchen,Kitchen Appliances,787
80,Home & Kitchen,Cookware,736
82,Home & Kitchen,Furniture,736
78,Home & Kitchen,Bedding,684
77,Home & Kitchen,Bathroom Accessories,683
83,Home & Kitchen,Home Lighting,681
79,Home & Kitchen,Cleaning Supplies,681
81,Home & Kitchen,Dining & Serveware,680
18,Beauty & Personal Care,Skincare,679
44,Electronics,Mobile Accessories,679


In [13]:
# ================================
# CELL 10 — SUBCATEGORY TF-IDF
# ================================

subcategory_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    max_features=300000
)

subcategory_vectorizer.fit(product_df["text"])

print("✅ Subcategory vectorizer trained.")
print(
    "Vocabulary size:",
    len(subcategory_vectorizer.vocabulary_)
)

✅ Subcategory vectorizer trained.
Vocabulary size: 52494


In [14]:
# ================================
# CELL 11 — TRAIN SUBCATEGORY MODELS
# ================================

subcategory_models = {}
subcategory_single_labels = {}

for category in sorted(product_df["category"].unique()):

    category_data = product_df[
        product_df["category"] == category
    ].copy()

    # Subcategory counts
    counts = category_data["subcategory"].value_counts()

    # Remove extremely rare labels
    valid_labels = counts[counts >= 2].index

    category_data = category_data[
        category_data["subcategory"].isin(valid_labels)
    ].copy()

    unique_subcategories = category_data[
        "subcategory"
    ].unique()

    print(
        f"\n{category}: "
        f"{len(category_data)} records | "
        f"{len(unique_subcategories)} subcategories"
    )

    # Only one subcategory
    if len(unique_subcategories) == 1:

        subcategory_single_labels[category] = (
            unique_subcategories[0]
        )

        print(
            "→ Single label:",
            unique_subcategories[0]
        )

        continue

    X_sub = category_data["text"]
    y_sub = category_data["subcategory"]

    # Train/test split
    X_sub_train, X_sub_test, y_sub_train, y_sub_test = train_test_split(
        X_sub,
        y_sub,
        test_size=0.20,
        random_state=42,
        stratify=y_sub
    )

    # TF-IDF
    X_sub_train_tfidf = subcategory_vectorizer.transform(
        X_sub_train
    )

    # Train model
    model = LinearSVC(
        C=2.0,
        class_weight="balanced",
        random_state=42
    )

    model.fit(
        X_sub_train_tfidf,
        y_sub_train
    )

    subcategory_models[category] = model

print("\n================================")
print("✅ SUBCATEGORY TRAINING COMPLETE")
print("================================")
print("Category models:", len(subcategory_models))
print(
    "Single-label categories:",
    len(subcategory_single_labels)
)


Automotive: 3129 records | 5 subcategories

Baby: 3448 records | 6 subcategories

Beauty & Personal Care: 4902 records | 8 subcategories

Books & Stationery: 3496 records | 6 subcategories

Clothing: 8804 records | 15 subcategories

Electronics: 6222 records | 10 subcategories

Footwear: 5169 records | 9 subcategories

Garden & Outdoor: 2861 records | 5 subcategories

Grocery: 5101 records | 8 subcategories

Health & Wellness: 1851 records | 5 subcategories

Home & Kitchen: 7026 records | 10 subcategories

Home Decor: 7842 records | 15 subcategories

Jewelry & Accessories: 4064 records | 7 subcategories

Medical & Pharmacy: 12687 records | 35 subcategories

Musical Instruments: 2862 records | 5 subcategories

Office Products: 2869 records | 5 subcategories

Pet Supplies: 3441 records | 6 subcategories

Sports & Fitness: 4328 records | 7 subcategories

Tools & Home Improvement: 3488 records | 6 subcategories

Toys & Games: 4118 records | 7 subcategories

Travel & Luggage: 2292 records 

In [15]:
# ================================
# CELL 12 — SUBCATEGORY EVALUATION
# ================================

subcategory_results = []

for category, model in subcategory_models.items():

    category_data = product_df[
        product_df["category"] == category
    ].copy()

    counts = category_data["subcategory"].value_counts()

    valid_labels = counts[counts >= 2].index

    category_data = category_data[
        category_data["subcategory"].isin(valid_labels)
    ]

    X_sub = category_data["text"]
    y_sub = category_data["subcategory"]

    X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(
        X_sub,
        y_sub,
        test_size=0.20,
        random_state=42,
        stratify=y_sub
    )

    X_test_sub_tfidf = subcategory_vectorizer.transform(
        X_test_sub
    )

    predictions = model.predict(
        X_test_sub_tfidf
    )

    acc = accuracy_score(
        y_test_sub,
        predictions
    )

    subcategory_results.append({
        "Category": category,
        "Accuracy": round(acc, 4),
        "Test Records": len(y_test_sub),
        "Subcategories": y_test_sub.nunique()
    })

subcategory_results_df = pd.DataFrame(
    subcategory_results
)

display(
    subcategory_results_df.sort_values(
        "Accuracy",
        ascending=False
    )
)

,Category,Accuracy,Test Records,Subcategories
0,Automotive,1.0,626,5
1,Baby,1.0,690,6
2,Beauty & Personal Care,1.0,981,8
3,Books & Stationery,1.0,700,6
4,Clothing,1.0,1761,15
5,Electronics,1.0,1245,10
6,Footwear,1.0,1034,9
7,Garden & Outdoor,1.0,573,5
8,Grocery,1.0,1021,8
9,Health & Wellness,1.0,371,5


In [16]:
# ================================
# CELL 13 — SAVE SUBCATEGORY MODELS
# ================================

import os
import joblib

MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(
    subcategory_models,
    f"{MODEL_DIR}/subcategory_models.pkl"
)

joblib.dump(
    subcategory_single_labels,
    f"{MODEL_DIR}/subcategory_single_labels.pkl"
)

joblib.dump(
    subcategory_vectorizer,
    f"{MODEL_DIR}/subcategory_vectorizer.pkl"
)

print("✅ Subcategory models saved.")
print("\nSaved files:")

print("subcategory_models.pkl")
print("subcategory_single_labels.pkl")
print("subcategory_vectorizer.pkl")

✅ Subcategory models saved.

Saved files:
subcategory_models.pkl
subcategory_single_labels.pkl
subcategory_vectorizer.pkl


In [45]:
# ================================
# CELL 14 — COMPLETE PREDICTION TEST
# ================================

def predict_product(title, description=""):

    text = f"{title} {description}".strip()

    # Category
    category_features = category_vectorizer.transform([text])

    category = category_model.predict(
        category_features
    )[0]

    # Not a Product
    if category == "Not a Product":
        return {
            "category": "Not a Product",
            "subcategory": "Not a Product"
        }

    # Single subcategory
    if category in subcategory_single_labels:

        subcategory = subcategory_single_labels[
            category
        ]

    # Category-specific model
    elif category in subcategory_models:

        sub_features = subcategory_vectorizer.transform(
            [text]
        )

        subcategory = subcategory_models[
            category
        ].predict(sub_features)[0]

    else:
        subcategory = "Unknown"

    return {
        "category": category,
        "subcategory": subcategory
    }


test_products = [
    ("Yahya is Founder", ""),
    ("Good morning everyone", ""),
    ("Apple iPhone 15 128GB Smartphone", ""),
    ("Wireless Bluetooth Headphones", "with charging case"),
    ("Nike Men's Running Shoes", ""),
    ("Digital Blood Pressure Monitor", ""),
    ("Paracetamol 500mg Tablets", "")
]

for title, description in test_products:

    result = predict_product(
        title,
        description
    )

    print("\nInput:", title)
    print("Category:", result["category"])
    print("Subcategory:", result["subcategory"])


Input: Yahya is Founder
Category: Not a Product
Subcategory: Not a Product

Input: Good morning everyone
Category: Not a Product
Subcategory: Not a Product

Input: Apple iPhone 15 128GB Smartphone
Category: Electronics
Subcategory: Mobile Phones

Input: Wireless Bluetooth Headphones
Category: Electronics
Subcategory: Headphones

Input: Nike Men's Running Shoes
Category: Footwear
Subcategory: Running Shoes

Input: Digital Blood Pressure Monitor
Category: Not a Product
Subcategory: Not a Product

Input: Paracetamol 500mg Tablets
Category: Medical & Pharmacy
Subcategory: Pain Relief
